# 05 · Model Comparison & Champion Selection


Side-by-side comparison of all five models. Ranks by recall, precision, F1, 
and ROC-AUC. Selects the best-performing model as the champion, saves it as 
`xgboost_agrishield_v1.joblib`, and prepares it for production use in the 
FastAPI backend by verifying the feature contract and artifact integrity.


In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_DIR = Path("../../data_pipeline/data/processed")
MODEL_DIR = Path("../models")

# Load all models
models = {}
for name in ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM", "Linear Regression"]:
    safe_name = name.lower().replace(" ", "_")
    path = MODEL_DIR / f"{safe_name}.joblib"
    if path.exists():
        models[name] = joblib.load(path)
    else:
        print(f"Warning: {name} not found")

# Load evaluation data
y_test = pd.read_csv(DATA_DIR / "y_test.csv")["target_risk"]
feature_cols = json.load(open(MODEL_DIR / "feature_columns.json"))

# ─── Build Comparison Table ───
comparison = []
for name, model in models.items():
    if name == "Linear Regression":
        y_pred_proba = np.clip(model.predict(joblib.load(MODEL_DIR / "scaler.joblib").transform(pd.read_csv(DATA_DIR / "X_test.csv")[feature_cols])), 0, 1)
        y_pred = (y_pred_proba >= 0.5).astype(int)
    else:
        X_test = pd.read_csv(DATA_DIR / "X_test.csv")[feature_cols]
        y_pred = model.predict(X_test)
        y_pred_proba = model.predict_proba(X_test)[:, 1]

    comparison.append({
        "Model": name,
        "Accuracy": float((y_pred == y_test).mean()),
        "Precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "Recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "F1": float(f1_score(y_test, y_pred, zero_division=0)),
        "ROC-AUC": float(roc_auc_score(y_test, y_pred_proba)),
    })

comp_df = pd.DataFrame(comparison).sort_values("Recall", ascending=False)
print("MODEL COMPARISON (sorted by Recall)")
print(comp_df.round(4).to_string(index=False))

# ─── Visualize Comparison ───
metrics = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
fig, ax = plt.subplots(figsize=(12, 7))
x = np.arange(len(metrics))
width = 0.15
for i, (name, row) in enumerate(comp_df.iterrows()):
    ax.bar(x + i * width, [row[m] for m in metrics], width, label=name, alpha=0.85)
ax.set_xlabel("Metric", fontsize=12)
ax.set_ylabel("Score", fontsize=12)
ax.set_title("Model Performance Comparison", fontsize=14)
ax.set_xticks(x + width * (len(comp_df) - 1) / 2)
ax.set_xticklabels(metrics)
ax.legend(fontsize=9)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig(MODEL_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n💾 Comparison chart saved to models/model_comparison.png")

# ─── Select Champion ───
champion = comp_df.iloc[0]["Model"]
print(f"\n🏆 CHAMPION MODEL: {champion}")
print(f"   Recall: {comp_df.iloc[0]['Recall']:.4f}")
print(f"   F1:     {comp_df.iloc[0]['F1']:.4f}")
print(f"   ROC-AUC: {comp_df.iloc[0]['ROC-AUC']:.4f}")

# ─── Save Champion for Production ───
champion_model = models[champion]
champion_path = MODEL_DIR / "xgboost_agrishield_v1.joblib"
joblib.dump(champion_model, champion_path)
print(f"\n💾 Champion saved as: {champion_path}")

# Verify champion loads correctly
loaded = joblib.load(champion_path)
print(f"✅ Champion model re-loaded successfully")
print(f"   Type: {type(loaded).__name__}")
if hasattr(loaded, "feature_importances_"):
    print(f"   Features: {len(loaded.feature_importances_)}")
    top = sorted(zip(loaded.feature_importances_, feature_cols), reverse=True)[:5]
    print(f"   Top 5 features: {[f[1] for f in top]}")

# ─── Prepare Application Artifacts ───
print("\n── Application Artifact Check ──")
artifacts = [
    (MODEL_DIR / "xgboost_agrishield_v1.joblib", "Champion model"),
    (MODEL_DIR / "scaler.joblib", "Feature scaler"),
    (MODEL_DIR / "feature_columns.json", "Feature contract"),
    (MODEL_DIR / "county_ndvi_stats.csv", "County NDVI baselines"),
]
for path, desc in artifacts:
    exists = path.exists()
    size = path.stat().st_size if exists else 0
    status = "✅" if (exists and size > 0) else "❌"
    print(f"  {status} {desc}: {path.name} ({size:,} bytes)")

print("\n" + "=" * 80)
print("CHAMPION SELECTION COMPLETE")
print("=" * 80)
print(f"Model ready for production: {champion_path}")
